In [2]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("7 ways to create a DF"). \
getOrCreate()

CÁCH 1

In [4]:
df1 = spark.read \
.format("csv") \
.option("header", "true") \
.option("inferSchema", "true") \
.load("D:\Learn-spark\learn-spark-maide\orders_wh.csv")

In [5]:
df1.show(5)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
+--------+-------------------+-----------+---------------+
only showing top 5 rows



CÁCH 2

In [6]:
df1.createOrReplaceTempView("orders")

In [7]:
df2 = spark.sql("select * from orders")

In [8]:
df2.show(5)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
+--------+-------------------+-----------+---------------+
only showing top 5 rows



CÁCH 3

In [9]:
df3 = spark.table("orders")

In [10]:
df3.show(5)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
+--------+-------------------+-----------+---------------+
only showing top 5 rows



=>>>>>>> spark.sql linh hoạt hơn so với sql.table

CÁCH 4

In [11]:
df4 = spark.range(5)

In [12]:
df4.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



spark.range sử dụng trong trường hợp test, muốn tạo nhanh: để test với transformer nào đó, ..., ko mất công phải đọc từ file csv chẳng hạn

In [14]:
df4_1 = spark.range(1,10)
df4_1.show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [15]:
df4_2 = spark.range(1,8,2)
df4_2.show()

+---+
| id|
+---+
|  1|
|  3|
|  5|
|  7|
+---+



CÁCH 5: Tạo df từ 1 list

In [25]:
list_orders = [(1,'2013-07-25 00:00:00.0',11599,'CLOSED'), (2,'2013-07-25 00:00:00.0',256,'PENDING_PAYMENT'), (3,'2013-07-25 00:00:00.0',12111,'COMPLETE')]

In [26]:
df5 = spark.createDataFrame(list_orders)

In [27]:
df5.show()

+---+--------------------+-----+---------------+
| _1|                  _2|   _3|             _4|
+---+--------------------+-----+---------------+
|  1|2013-07-25 00:00:...|11599|         CLOSED|
|  2|2013-07-25 00:00:...|  256|PENDING_PAYMENT|
|  3|2013-07-25 00:00:...|12111|       COMPLETE|
+---+--------------------+-----+---------------+



In [28]:
df5.printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)
 |-- _3: long (nullable = true)
 |-- _4: string (nullable = true)



In [29]:
df5 = df5.toDF('order_id', 'order_date', 'cust_id', 'status')

In [30]:
df5.show()

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|         status|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
+--------+--------------------+-------+---------------+



CÁCH 6

In [31]:
order_schema = ['order_id', 'order_date', 'cust_id', 'status']

In [32]:
df6 = spark.createDataFrame(list_orders, order_schema)

In [33]:
df6.show()

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|         status|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
+--------+--------------------+-------+---------------+



In [37]:
df6.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- status: string (nullable = true)



Định nghĩa thêm kiểu dữ liệu cho nó

In [60]:
order_schema_1 = "'order_id' integer, 'order_date' string, 'cust_id' long, 'status' string"

In [64]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType

# Định nghĩa schema chuẩn
order_schema_2 = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("cust_id", LongType(), True),
    StructField("status", StringType(), True)
])

In [65]:
df61 = spark.createDataFrame(list_orders, order_schema_2)

In [66]:
df61.show()

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|         status|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
+--------+--------------------+-------+---------------+



In [38]:
df61.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- status: string (nullable = true)



CÁCH 7: Chỉ cần add thêm 1 cái schema cho 1 rdd cũng tạo thành 1 dataFrame mới

In [46]:
rdd = spark.sparkContext.textFile("D:\Learn-spark\learn-spark-maide\orders_wh.csv")

In [50]:
#Loại bỏ dòng schema (header) đầu
header = rdd.first()

In [51]:
data_rdd = rdd.filter(lambda row: row != header)

In [52]:
data_rdd.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE']

In [53]:
#Add schema
#đưa về dạng giống list_orders
order_rdd = data_rdd.map(lambda x: (int(x.split(",")[0]), x.split(",")[1], int(x.split(",")[2]), x.split(",")[3]))

In [54]:
order_rdd.take(5)

[(1, '2013-07-25 00:00:00.0', 11599, 'CLOSED'),
 (2, '2013-07-25 00:00:00.0', 256, 'PENDING_PAYMENT'),
 (3, '2013-07-25 00:00:00.0', 12111, 'COMPLETE'),
 (4, '2013-07-25 00:00:00.0', 8827, 'CLOSED'),
 (5, '2013-07-25 00:00:00.0', 11318, 'COMPLETE')]

In [67]:
# order_schema_7 = "'order_id' integer, 'order_date' string, 'cust_id' long, 'status' string"
order_schema_7 = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("cust_id", LongType(), True),
    StructField("status", StringType(), True)
])

In [68]:
df7 = spark.createDataFrame(order_rdd, order_schema_7)

In [69]:
df7.show()

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|         status|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
|       4|2013-07-25 00:00:...|   8827|         CLOSED|
|       5|2013-07-25 00:00:...|  11318|       COMPLETE|
|       6|2013-07-25 00:00:...|   7130|       COMPLETE|
|       7|2013-07-25 00:00:...|   4530|       COMPLETE|
|       8|2013-07-25 00:00:...|   2911|     PROCESSING|
|       9|2013-07-25 00:00:...|   5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:...|   5648|PENDING_PAYMENT|
|      11|2013-07-25 00:00:...|    918| PAYMENT_REVIEW|
|      12|2013-07-25 00:00:...|   1837|         CLOSED|
|      13|2013-07-25 00:00:...|   9149|PENDING_PAYMENT|
|      14|2013-07-25 00:00:...|   9842|     PROCESSING|
|      15|2013-07-25 00:00:...|   2568|       CO